In [1]:
!pip install cdsapi netCDF4 cfgrib xarray webdavclient3

  Using cached cdsapi-0.7.7-py2.py3-none-any.whl.metadata (3.1 kB)
  Using cached netcdf4-1.7.4-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (2.1 kB)
  Using cached cfgrib-0.9.15.1-py3-none-any.whl.metadata (56 kB)
  Using cached xarray-2026.1.0-py3-none-any.whl.metadata (12 kB)
  Using cached webdavclient3-3.14.7-py3-none-any.whl.metadata (10 kB)
  Using cached ecmwf_datastores_client-0.4.2-py3-none-any.whl.metadata (21 kB)
  Using cached cftime-1.6.5-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (8.7 kB)
  Using cached eccodes-2.45.0-py3-none-any.whl.metadata (15 kB)
  Using cached lxml-6.0.2-cp311-cp311-manylinux_2_26_x86_64.manylinux_2_28_x86_64.whl.metadata (3.6 kB)
  Using cached findlibs-0.1.2-py3-none-any.whl.metadata (4.5 kB)
  Using cached eccodeslib-2.45.1.9-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (4.2 kB)
  Using cached multiurl-0.3.7-py3-none-any.whl.metadata (2.8 kB)
  Using cached eckitlib-1.32.4.9-cp311-cp311-manylinu

In [3]:
import xarray as xr
import numpy as np
import pandas as pd
import gc
from pathlib import Path
from webdav3.client import Client
import shutil

In [4]:
# Check disk space
def check_space():
    total, used, free = shutil.disk_usage(".")
    free_gb = free / (1024**3)
    print(f"Total: {total/(1024**3):.1f} GB")
    print(f"Used: {used/(1024**3):.1f} GB")
    print(f"Free: {free_gb:.1f} GB")
    return free_gb

free_space = check_space()
if free_space < 5:
    print("WARNING: Less than 5GB free!")
else:
    print("Enough space to proceed!")

Total: 19.5 GB
Used: 15.6 GB
Free: 3.9 GB


In [5]:
DATA_FILE = Path('final_datasets/ocean_atmos_combined_DAILY_1993_2025.nc')
SUB_DATA_FILE = Path('final_datasets/subsurface_daily_1993_2025.nc')


In [6]:

print("Loading dataset with lazy loading...")
combined = xr.open_dataset(DATA_FILE, chunks={'time': 365})
ocean_native = xr.open_dataset(SUB_DATA_FILE, chunks={'time': 365})

print(f"Dataset loaded: {len(combined.time)} days, {len(combined.latitude)}×{len(combined.longitude)} grid")
print(f"Variables: {list(combined.data_vars)}")
print(f"\nMemory note: Data is lazily loaded. Features computed on-demand.")

Loading dataset with lazy loading...
Dataset loaded: 12053 days, 25×49 grid
Variables: ['temp_0_50m', 'temp_50_100m', 'temp_100_150m', 'ohc_0_50m', 'ohc_50_100m', 'ohc_100_150m', 'u10', 'v10', 't2m', 'msl', 'sst', 'wind_speed', 'ssr_flux', 'str_flux', 'sensible_flux', 'latent_flux', 'net_heat_flux']

Memory note: Data is lazily loaded. Features computed on-demand.


/tmp/ipykernel_304/3066873907.py:2: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  combined = xr.open_dataset(DATA_FILE, chunks={'time': 365})


In [7]:
def validate_coverage(combined):
    """
    Validate the merged dataset structure and coverage
    """
    print("\n Validating merged dataset...")
    
    # Check ocean mask
    ocean_mask = combined['temp_0_50m'].isel(time=0).notnull()
    ocean_points = ocean_mask.sum().values
    total_points = ocean_mask.size
    
    print(f"   Grid points:")
    print(f"      Ocean: {ocean_points} ({ocean_points/total_points*100:.1f}%)")
    print(f"      Land: {total_points - ocean_points} ({(total_points-ocean_points)/total_points*100:.1f}%)")
    
    # Check sample location (center of domain - likely ocean)
    sample_lon = combined.longitude.values[len(combined.longitude)//2]
    sample_lat = combined.latitude.values[len(combined.latitude)//2]
    sample = combined.sel(longitude=sample_lon, latitude=sample_lat, time=combined.time[0])
    
    print(f"\n   Sample point ({sample_lat:.1f}°N, {sample_lon:.1f}°E):")
    check_vars = ['temp_0_50m', 'sst', 't2m', 'ssr_flux']
    for var in check_vars:
        if var in sample:
            has_data = not np.isnan(sample[var].values)
            status = 'okay' if has_data else 'not okay'
            print(f"      {status} {var}")
    
    # Overall coverage (sample first timestep to avoid loading all data)
    print(f"\n   Data coverage (first timestep):")
    first_time = combined.isel(time=0)
    for var in combined.data_vars:
        valid_pct = (first_time[var].notnull().sum() / first_time[var].size * 100).values
        print(f"      {var}: {valid_pct:.1f}% valid")
    
    print(f"\n Validation complete")
    
    return True



In [8]:
def check_physical_ranges(combined):
    """Check if values are physically realistic"""
    print("\n Physical Range Validation:")
    
    # UPDATED ranges for Mediterranean
    ranges = {
        'temp_0_50m': (10, 33),      # Extended for Med summer extremes
        'temp_50_100m': (12, 30),
        'temp_100_150m': (13, 28),
        'sst': (10, 34),             # Extended for heatwaves
        't2m': (-25, 50),            # °C (or use 250-320 if in Kelvin)
        'u10': (-30, 30),
        'v10': (-30, 30),
        'msl': (95000, 105000),
    }
    
    issues = []
    for var, (vmin, vmax) in ranges.items():
        if var in combined:
            data = combined[var]
            actual_min = float(data.min())
            actual_max = float(data.max())
            
            if actual_min < vmin or actual_max > vmax:
                issues.append(f" issue - {var}: [{actual_min:.2f}, {actual_max:.2f}] outside [{vmin}, {vmax}]")
                print(f" issue - {var}: range [{actual_min:.2f}, {actual_max:.2f}]")
            else:
                print(f" okay {var}: range [{actual_min:.2f}, {actual_max:.2f}]")
    
    return len(issues) == 0, issues

In [9]:
def compare_with_native(ocean_native, ocean_regridded, era5):
    """
    Compare regridded ocean data with native resolution
    """
    print("\n Regridding Accuracy Assessment:")
    
    # Regrid native back to itself at a common sample location
    # to check interpolation artifacts
    
    from scipy import stats
    
    for var in ['temp_0_50m', 'temp_50_100m', 'temp_100_150m']:
        # Sample both on overlapping grid points
        native_sample = ocean_native[var].isel(time=slice(0, 10)).values.flatten()
        regrid_sample = ocean_regridded[var].isel(time=slice(0, 10)).values.flatten()
        
        # Remove NaNs
        mask = ~(np.isnan(native_sample) | np.isnan(regrid_sample))
        native_clean = native_sample[mask]
        regrid_clean = regrid_sample[mask]
        
        if len(native_clean) > 0:
            bias = np.mean(regrid_clean - native_clean)
            rmse = np.sqrt(np.mean((regrid_clean - native_clean)**2))
            corr = stats.pearsonr(native_clean, regrid_clean)[0]
            
            print(f"   {var}:")
            print(f"      Bias: {bias:.4f} °C")
            print(f"      RMSE: {rmse:.4f} °C")
            print(f"      Correlation: {corr:.4f}")

In [10]:
def plot_spatial_validation(combined, var='temp_0_50m', timestep=0):
    """
    Create validation plot for supplementary material
    """
    import matplotlib.pyplot as plt
    import cartopy.crs as ccrs
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), 
                             subplot_kw={'projection': ccrs.PlateCarree()})
    
    data = combined[var].isel(time=timestep)
    
    # Plot 1: Raw field
    axes[0].pcolormesh(combined.longitude, combined.latitude, 
                       data, transform=ccrs.PlateCarree())
    axes[0].coastlines()
    axes[0].set_title(f'{var} - Merged Product')
    
    # Plot 2: Data coverage mask
    mask = data.notnull()
    axes[1].pcolormesh(combined.longitude, combined.latitude, 
                       mask, transform=ccrs.PlateCarree(), cmap='binary')
    axes[1].coastlines()
    axes[1].set_title('Data Coverage')
    
    # Plot 3: Histogram
    axes[2].hist(data.values.flatten(), bins=50, edgecolor='black')
    axes[2].set_xlabel(f'{var} (°C)')
    axes[2].set_ylabel('Frequency')
    axes[2].set_title('Value Distribution')
    
    plt.tight_layout()


In [11]:
def check_temporal_continuity(combined):
    """Check for gaps or jumps in time series"""
    print("\n Temporal Continuity Check:")
    
    time_diff = combined.time.diff('time')
    expected_diff = np.timedelta64(1, 'D')
    
    gaps = (time_diff != expected_diff).sum().values
    
    if gaps > 0:
        print(f" issue - Found {gaps} time gaps")
    else:
        print(f"okay - No time gaps (continuous daily data)")
    
    # UPDATED thresholds
    thresholds = {
        'temp_0_50m': 5.0,   # Increased - upwelling can cause this
        'sst': 5.0,
        't2m': 15.0,         # Much higher - air temp is variable!
    }
    
    for var, threshold in thresholds.items():
        if var in combined:
            temp_diff = combined[var].diff('time')
            extreme_changes = (np.abs(temp_diff) > threshold).sum().values
            total_changes = temp_diff.size
            pct = (extreme_changes / total_changes * 100) if total_changes > 0 else 0
            
            # Only flag if >1% of data shows extreme changes
            if pct > 1.0:
                print(f" issue - {var}: {extreme_changes} extreme changes ({pct:.2f}% > {threshold}°)")
            else:
                print(f" okay - {var}: {extreme_changes} changes >>{threshold}° ({pct:.3f}% - acceptable)")

In [12]:
def verify_coordinate_alignment(combined):
    """Verify all variables share exact coordinates"""
    print("\n Coordinate Alignment Check:")
    
    ref_lon = combined.longitude.values
    ref_lat = combined.latitude.values
    ref_time = combined.time.values
    
    all_aligned = True
    for var in combined.data_vars:
        lon_match = np.allclose(combined[var].longitude.values, ref_lon)
        lat_match = np.allclose(combined[var].latitude.values, ref_lat)
        time_match = np.all(combined[var].time.values == ref_time)
        
        if not (lon_match and lat_match and time_match):
            print(f"   ⚠ {var}: coordinates MISALIGNED")
            all_aligned = False
    
    if all_aligned:
        print(f" All variables share identical coordinates")
    
    return all_aligned

In [13]:
def validate_merged_dataset(combined, ocean_native=None):
    """
    Comprehensive validation of merged ocean-atmosphere dataset
    """
    print("="*60)
    print("DATASET VALIDATION REPORT")
    print("="*60)
    
    # 1. original checks
    validate_coverage(combined)  # current function
    
    # 2. Physical range checks
    ranges_ok, issues = check_physical_ranges(combined)
    
    # 3. Coordinate alignment
    coords_ok = verify_coordinate_alignment(combined)
    
    # 4. Temporal continuity
    check_temporal_continuity(combined)
    
    # 5. If native data available, compare
    if ocean_native is not None:
        compare_with_native(ocean_native, combined, None)
    
    # 6. Final summary
    print("\n" + "="*60)
    if ranges_ok and coords_ok:
        print("VALIDATION PASSED - Dataset ready for analysis")
    else:
        print("VALIDATION WARNINGS - Review issues above")
    print("="*60)
    
    return ranges_ok and coords_ok

In [14]:
validate_merged_dataset(combined)

DATASET VALIDATION REPORT

 Validating merged dataset...
   Grid points:
      Ocean: 907 (74.0%)
      Land: 318 (26.0%)

   Sample point (34.0°N, 30.0°E):
      okay temp_0_50m
      okay sst
      okay t2m
      okay ssr_flux

   Data coverage (first timestep):
      temp_0_50m: 74.0% valid
      temp_50_100m: 71.5% valid
      temp_100_150m: 69.7% valid
      ohc_0_50m: 74.0% valid
      ohc_50_100m: 71.5% valid
      ohc_100_150m: 69.7% valid
      u10: 100.0% valid
      v10: 100.0% valid
      t2m: 100.0% valid
      msl: 100.0% valid
      sst: 79.3% valid
      wind_speed: 100.0% valid
      ssr_flux: 100.0% valid
      str_flux: 100.0% valid
      sensible_flux: 100.0% valid
      latent_flux: 100.0% valid
      net_heat_flux: 100.0% valid

 Validation complete

 Physical Range Validation:
 okay temp_0_50m: range [13.39, 32.04]
 okay temp_50_100m: range [13.46, 28.79]
 okay temp_100_150m: range [13.47, 23.60]
 okay sst: range [12.24, 33.27]
 okay t2m: range [-18.64, 39.58]
 o

True